In [33]:
import os
# Set your API keys as environment variables or in a .env file
# os.environ["HUGGING_FACE_KEY"] = "your_hugging_face_key_here"
# os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"



install libs

In [4]:
%pip install -q youtube-transcript-api langchain-community langchain-huggingface \
faiss-cpu tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install -q langchain-text-splitters


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEndpoint,HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate








C:\Users\Samarth\AppData\Local\Temp\ipykernel_28032\4092164627.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


STEP 1-- INDEXING

In [49]:
##LOADING TRANSCRIPTT


video_id = "bzEO6RngvL4"  # only the ID, not full URL
#YouTubeTranscriptApi=YouTubeTranscriptApi()


try:
    # If you don't care which language, this returns the "best" one
    transcript_list = YouTubeTranscriptApi.fetch(
        video_id, languages=["en"]
    )    #transcipt list contain three things text,startitme and duration

    # Flatten it to plain text
    transcript = " ".join(
        chunk.text for chunk in transcript_list
    )
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

It only takes 3 months to completely change the trajectory of your life. In this video, I'm going to break down seven steps that I've used throughout my career that you guys can apply how to build your goals, how to build systems to hit those goals, and actually build the life that you want. The hardest thing to understand at the beginning is that if you're chasing after a goal with the intention of achieving self-worth or of proving your worth, then that's the kind of goal where when you finish it, when you get it, you're left feeling empty. Every meaningful goal has a sacrifice. That's why you want to make sure that what you're sacrificing for is actually worth it and more important to you than the things you're leaving behind. >> [snorts] >> So, how do you decide whether a goal is worth pursuing? >> I've spoken about this in a video in the past, but one of the biggest exercises you can do to make sure you're on the right goals and on the right path is defining what your values are. 

In [50]:
len(transcript)

17833

In [51]:
## CREATE CHUNKS (TEXT SPLITTER)

splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([transcript])

In [52]:
len(chunks)

23

In [53]:
chunks[0]

Document(metadata={}, page_content="It only takes 3 months to completely change the trajectory of your life. In this video, I'm going to break down seven steps that I've used throughout my career that you guys can apply how to build your goals, how to build systems to hit those goals, and actually build the life that you want. The hardest thing to understand at the beginning is that if you're chasing after a goal with the intention of achieving self-worth or of proving your worth, then that's the kind of goal where when you finish it, when you get it, you're left feeling empty. Every meaningful goal has a sacrifice. That's why you want to make sure that what you're sacrificing for is actually worth it and more important to you than the things you're leaving behind. >> [snorts] >> So, how do you decide whether a goal is worth pursuing? >> I've spoken about this in a video in the past, but one of the biggest exercises you can do to make sure you're on the right goals and on the right pat

In [54]:
 #%pip install sentence-transformers

In [55]:
## EMBEDDING GEN AND STORING IN VECTOR STORE


embeddings = HuggingFaceEmbeddings(
    model_name="dmis-lab/UltraBERT-Embed-150M"
)
vector_store=FAISS.from_documents(chunks,embeddings)  ## store embedding vectors

Loading weights: 100%|██████████| 218/218 [00:00<00:00, 378.53it/s]


In [56]:
print("Total vectors:", vector_store.index.ntotal)


Total vectors: 23


STEP 2-- Retrieval


In [57]:
retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [58]:
retrieved_dpcs=retriever.invoke("What to do in this challenge?")

STEP--3  AUGMENTATION

In [59]:
import os
# Set your API keys as environment variables or in a .env file
# os.environ["HUGGING_FACE_KEY"] = "your_hugging_face_key_here"
# os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"


In [60]:
prompt=PromptTemplate(
    template=""" 
        You are a helpful assistant.
        Answer ONLY from the provided Transcript context.
        If the context is insufficient just simply say I dont know.

        {context}
        Question: {Question}
    
    
    """,
    input_variables=['context','Question']
)

In [61]:
retrieved_dpcs

[Document(id='7e645a38-a923-4b9c-abb0-6c19139329d3', metadata={}, page_content="some point, and when you need to fall back on what's your foundation. >> What process [snorts] should someone watching this follow to set standards for their goals? >> I would say easy thing to start with is to pick three absolute non-negotiables that you know you can do no matter what. I want to get jacked, I'm going to train at least three times a week because no matter how much I'm traveling, how little sleep I get, I'm I'm to train [music] three days a week. So, it's not let me build a huge list of things that I'm overwhelmed with. What are three things I can decide right now that I'm absolutely never going to falter on? And when you fall short, how are you going to respond? The goal of this isn't to become someone who never makes mistakes. It's to become someone who makes as minimal mistakes as possible. And when they do they make a mistake, they're not defined by them. They get back up and they're abl

In [ ]:
question="How many steps are told in this video"
retrieved_doc=retriever.invoke(question)

[Document(id='b03db0f6-46b8-46f6-b128-a974266e7593', metadata={}, page_content="and not for others. If you guys liked this video, we have another similar video I can link right here. This video goes more in depth of more mindset stuff. As always, appreciate you guys for watching, and I'll see you next time."),
 Document(id='7e645a38-a923-4b9c-abb0-6c19139329d3', metadata={}, page_content="some point, and when you need to fall back on what's your foundation. >> What process [snorts] should someone watching this follow to set standards for their goals? >> I would say easy thing to start with is to pick three absolute non-negotiables that you know you can do no matter what. I want to get jacked, I'm going to train at least three times a week because no matter how much I'm traveling, how little sleep I get, I'm I'm to train [music] three days a week. So, it's not let me build a huge list of things that I'm overwhelmed with. What are three things I can decide right now that I'm absolutely n

In [76]:
context="\n\n".join(doc.page_content for doc in retrieved_doc)  ## create a string containg text of all retrieved doc

In [77]:
final_prompt=prompt.invoke({"context":context,"Question":question})

In [78]:
%pip install -q langchain-groq


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


STEP 4-- GENERATION

In [79]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model_name="openai/gpt-oss-120b",
    temperature=0.2
)

# Test invocation with your prompt
answer = model.invoke(final_prompt)
print(answer.content)


Five steps are outlined in the video.


BUILDING A CHAIN

In [80]:
#till now we had to run each step manually chain solves this problem

from langchain_core.runnables import RunnableParallel,RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser



 

In [81]:
# define fucntion to make large context string for runnable lambda

def format_doc(retrieved):
    context_text="\n\n".join(doc.page_content for doc in retrieved)
    return context_text

In [86]:
parallel_chain=RunnableParallel({
    'context':retriever|RunnableLambda(format_doc),
    'Question':RunnablePassthrough()
})

In [87]:
parser=StrOutputParser()
main_chain=parallel_chain|prompt|model|parser

In [88]:
main_chain.invoke("Can you summarise this video")

'The video is about creating practical, sustainable habits and mind‑sets for achieving goals. The speaker suggests:\n\n1. **Pick three non‑negotiable actions** you can commit to no matter how busy or tired you are (e.g., train three times a week). Keep the list short so it isn’t overwhelming.  \n2. **Plan how you’ll respond when you fall short**—the aim isn’t to be perfect, but to make as few mistakes as possible and recover quickly without letting them define you.  \n3. **Address “too busy/tired” excuses** by pointing out that avoiding the work now only makes future fatigue worse; doing the hard work now makes later life easier.  \n4. **Use visualization**: repeatedly imagine yourself performing the desired behavior while feeling confident; this builds a mental‑physical connection that makes it easier to enter that state when needed.  \n5. **Track data and progress**: avoid getting stuck in hustle culture by regularly checking whether your effort is actually moving you toward your goa